[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/06_dpo_preference_pairs.ipynb)

# Step 6 — DPO Preference Pairs (Optional)

Generate **preference pairs** for Direct Preference Optimization (DPO) that teach
a small model *refusal vs engagement calibration* on the **SEC investor bulletin**
(scope-boundary document).

SFT (notebook 05) copies a single gold answer. DPO instead says: for the same
question, **this** answer is better than **that** one — which is the right tool
when the failure is a tradeoff (too helpful vs too silent, or inventing SEC power).

## Learning objectives
- Build boundary questions from train-split SEC paragraphs only (no test leak)
- Generate four labeled candidates per question (one chosen, three rejected)
- Expand candidates into TRL-style `{prompt, chosen, rejected}` rows
- Optionally run LoRA DPO when CUDA is available

## Prerequisites

1. Run **notebook 01** so `data/paragraphs.jsonl` exists (train/test splits).
2. Teacher API key in `implementations/qa_text_generation/.env` (same as notebooks 02–04).
3. Optional DPO train cell: `uv sync --group text-sft` and `RUN_DPO=1` on an NVIDIA GPU.

This notebook is **SEC-only**. It does not use the CFPB credit-card agreement.

In [1]:
import os
from pathlib import Path

from aieng.syn_data.text import (
    PARAGRAPHS_PATH,
    Paragraph,
    create_judge_client,
    create_teacher_client,
    load_implementation_dotenv,
    load_typed_jsonl,
    save_typed_jsonl,
    use_repo_root,
)
from aieng.syn_data.text.dpo import (
    DEFAULT_DPO_QUESTIONS,
    DPO_ADAPTER_DIR,
    DPO_CANDIDATES_PATH,
    DPO_PAIRS_PATH,
    CalibrationPrompt,
    PreferencePair,
    candidates_to_dpo_pairs,
    filter_pairs_with_judge,
    filter_sec_train_paragraphs,
    generate_boundary_prompts,
    generate_calibration_candidates,
    summarize_rejected_kinds,
    train_lora_dpo,
)
from rich.console import Console
from rich.table import Table


load_implementation_dotenv()
use_repo_root(Path("."))

N_QUESTIONS = int(os.getenv("DPO_N_QUESTIONS", DEFAULT_DPO_QUESTIONS))
VALIDATE_WITH_JUDGE = os.getenv("VALIDATE_WITH_JUDGE", "0") == "1"
RUN_DPO = os.getenv("RUN_DPO", "0") == "1"
SFT_BASE_MODEL = os.getenv("SFT_BASE_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
BASE_MODEL = os.getenv("DPO_BASE_MODEL", SFT_BASE_MODEL)

console = Console(width=100)
console.print(
    f"N_QUESTIONS={N_QUESTIONS}  VALIDATE_WITH_JUDGE={VALIDATE_WITH_JUDGE}  "
    f"RUN_DPO={RUN_DPO}\nBASE_MODEL={BASE_MODEL}  SFT_BASE_MODEL={SFT_BASE_MODEL}"
)

N_QUESTIONS=8  VALIDATE_WITH_JUDGE=True  RUN_DPO=True
BASE_MODEL=Qwen/Qwen2.5-0.5B-Instruct  SFT_BASE_MODEL=Qwen/Qwen2.5-0.5B-Instruct

## Why four candidates?

The SEC bulletin is **investor education** (passphrases, alerts, public Wi-Fi). It does
not tell anyone which stock to buy, and it does not turn optional tips into legal mandates.

For each boundary question we ask the teacher for **one JSON object** with four answers:

| Kind | DPO role | What it does wrong (or right) |
|------|----------|-------------------------------|
| `correctly_scoped` | **chosen** | Answers from the passage; hedges; refuses investment advice |
| `overreaching` | rejected | Gives buy/sell or personal advice the bulletin does not authorize |
| `underreaching` | rejected | Refuses a question the passage *could* answer |
| `authority_misattribution` | rejected | Claims the SEC requires or covers something it does not |

Each question expands to **three** DPO rows (chosen vs each rejected kind).

## 1. Load SEC train paragraphs

Use the **train** split only. Using `test_set.jsonl` here would leak the evaluation set
into preference training.

In [2]:
# Load the previously generated paragraphs.jsonl
all_paragraphs = load_typed_jsonl(PARAGRAPHS_PATH, Paragraph.from_dict)
# Filter the paragraphs to only include the SEC train set
sec_train = filter_sec_train_paragraphs(all_paragraphs)

if not sec_train:
    raise FileNotFoundError(
        f"No SEC train paragraphs in {PARAGRAPHS_PATH}. Run notebook 01 first."
    )

table = Table(title="SEC train paragraphs (scope-boundary)")
table.add_column("#", justify="right")
table.add_column("para_id")
table.add_column("chars", justify="right")
table.add_column("preview")
for i, paragraph in enumerate(sec_train[:8]):
    preview = paragraph.text.replace("\n", " ")[:80] + "…"
    table.add_row(str(i), paragraph.para_id, str(len(paragraph.text)), preview)
console.print(table)
console.print(f"Total SEC train paragraphs: {len(sec_train)}")

                               SEC train paragraphs (scope-boundary)                                
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ para_id                      ┃ chars ┃ preview                                               ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 0 │ sec_investor_bulletin::p0000 │   752 │ Updated Investor Bulletin: Protecting Your Online     │
│   │                              │       │ Investment Accounts from Fraud…                       │
│ 1 │ sec_investor_bulletin::p0001 │   795 │ Consider using a “strong” passphrase, instead of a    │
│   │                              │       │ password, if available. Passp…                        │
│ 2 │ sec_investor_bulletin::p0002 │   570 │ If you can’t use a passphrase, pick a “strong”        │
│   │                              │       │ password, keep it secure, and cha…                    │
│ 3 │ sec_investor_bulletin::p0003 │   464 │ What are passkeys? Some investment account websites   │
│   │                              │       │ have started using what is k…                         │
│ 4 │ sec_investor_bulletin::p0007 │   611 │ Account logins Failed account login attempts Password │
│   │                              │       │ changes Personal informati…                           │
│ 5 │ sec_investor_bulletin::p0008 │   512 │ Add biometric safeguards, if available. Your          │
│   │                              │       │ brokerage firm or investment advise…                  │
│ 6 │ sec_investor_bulletin::p0010 │   244 │ Avoid using public computers to access your           │
│   │                              │       │ investment accounts. Avoid accessing…                 │
│ 7 │ sec_investor_bulletin::p0011 │   744 │ Avoid using public computers that require you to      │
│   │                              │       │ enter personal information in o…                      │
└───┴──────────────────────────────┴───────┴───────────────────────────────────────────────────────┘

Total SEC train paragraphs: 11

## 2. Generate boundary questions

The teacher cycles three question types: **in-scope** (should answer with a hedge),
**out-of-scope** (should refuse investment advice), and **gray-boundary** (related to
the bulletin but easy to overclaim SEC authority).

Demo size is small (`N_QUESTIONS`, default 8). Increase it for a real corpus.

In [3]:
teacher = create_teacher_client()
boundary_prompts = generate_boundary_prompts(
    teacher, sec_train, n_questions=N_QUESTIONS
)

q_table = Table(title="Boundary questions")
q_table.add_column("id")
q_table.add_column("kind")
q_table.add_column("question")
for prompt in boundary_prompts:
    q_table.add_row(prompt.id, prompt.question_kind.value, prompt.question[:90])
console.print(q_table)

                                         Boundary questions                                         
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ id           ┃ kind          ┃ question                                                          ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ dpo-ff9d1d00 │ in_scope      │ What are some steps recommended by the SEC to safeguard personal  │
│              │               │ financial information for                                         │
│ dpo-792a7d6c │ out_of_scope  │ Can you recommend a specific investment platform that supports    │
│              │               │ strong passphrases and also                                       │
│ dpo-91b210ec │ gray_boundary │ Does the SEC legally require investors to change their investment │
│              │               │ account passwords every                                           │
│ dpo-4a26d678 │ in_scope      │ What is the difference between a passkey and a password when      │
│              │               │ accessing investment account                                      │
│ dpo-14b26126 │ out_of_scope  │ Can you recommend which investment firm offers the best online    │
│              │               │ account alerts for managing                                       │
│ dpo-d5e0e573 │ gray_boundary │ Does the SEC legally require all brokerage firms to implement     │
│              │               │ biometric safeguards like fi                                      │
│ dpo-9c8054c0 │ in_scope      │ What precautions should you take if you must use a public         │
│              │               │ computer to access your investme                                  │
│ dpo-216c6326 │ out_of_scope  │ Can you recommend a reliable broker for managing my investment    │
│              │               │ accounts securely, consider                                       │
└──────────────┴───────────────┴───────────────────────────────────────────────────────────────────┘

## 3. Generate four candidates per question

One teacher call returns all four answers so the contrast is internally consistent.

In [4]:
calibration_prompts: list[CalibrationPrompt] = []
for prompt in boundary_prompts:
    try:
        calibration_prompts.append(generate_calibration_candidates(teacher, prompt))
    except (KeyError, ValueError, TypeError, RuntimeError) as exc:
        console.print(f"[yellow]Skipping {prompt.id}: {type(exc).__name__}: {exc}[/yellow]")

example = next(
    (item for item in calibration_prompts if item.candidates),
    None,
)
if example is None:
    raise RuntimeError("Teacher returned no candidates. Check the API key and model.")

console.print(f"[bold]Example question[/bold] ({example.question_kind.value}):")
console.print(example.question)

cand_table = Table(title=f"Candidates for {example.id}")
cand_table.add_column("kind", style="cyan")
cand_table.add_column("role")
cand_table.add_column("answer preview")
cand_table.add_column("rationale")
for candidate in example.candidates:
    role = "chosen" if candidate.kind.value == "correctly_scoped" else "rejected"
    cand_table.add_row(
        candidate.kind.value,
        role,
        candidate.answer.replace("\n", " ")[:180],
        candidate.rationale[:180],
    )
console.print(cand_table)

Example question (in_scope):

What are some steps recommended by the SEC to safeguard personal financial information for online 
investment accounts?

                                    Candidates for dpo-ff9d1d00                                     
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ kind                     ┃ role     ┃ answer preview              ┃ rationale                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ correctly_scoped         │ chosen   │ The SEC recommends          │ The passage provides         │
│                          │          │ safeguarding personal       │ specific examples of         │
│                          │          │ financial information such  │ personal financial           │
│                          │          │ as Social Security numbers, │ information that should be   │
│                          │          │ financial account numbers,  │ protected, as recommended by │
│                          │          │ phone numbers, e-mail       │ the SEC.                     │
│                          │          │ addresses, and usernames    │                              │
│                          │          │ and passwo                  │                              │
│ overreaching             │ rejected │ You should definitely       │ The passage does not provide │
│                          │          │ invest in cybersecurity     │ specific investment advice   │
│                          │          │ software to protect your    │ or guarantee the             │
│                          │          │ online investment accounts, │ effectiveness of particular  │
│                          │          │ as this will guarantee your │ products.                    │
│                          │          │ information is safe.        │                              │
│ underreaching            │ rejected │ The bulletin does not       │ The passage does specify     │
│                          │          │ specify any steps to        │ steps, such as safeguarding  │
│                          │          │ safeguard personal          │ specific types of personal   │
│                          │          │ financial information.      │ financial information.       │
│ authority_misattribution │ rejected │ The SEC requires all        │ The passage does not state   │
│                          │          │ investors to use specific   │ that the SEC mandates        │
│                          │          │ cybersecurity measures to   │ specific cybersecurity       │
│                          │          │ protect their online        │ measures for all investors.  │
│                          │          │ accounts.                   │                              │
└──────────────────────────┴──────────┴─────────────────────────────┴──────────────────────────────┘

## 4. Expand to DPO rows

Prompts match notebooks 01 and 05 (`DEFAULT_EVAL_SYSTEM` + passage + question) so
preference training uses the same instruction format as SFT and evaluation.

In [5]:
pairs = candidates_to_dpo_pairs(calibration_prompts)
kind_counts = summarize_rejected_kinds(pairs)

count_table = Table(title="Preference pairs by rejected kind")
count_table.add_column("rejected kind")
count_table.add_column("n", justify="right")
for kind, count in kind_counts.items():
    count_table.add_row(kind, str(count))
console.print(count_table)
console.print(f"Total DPO rows: {len(pairs)}")

if pairs:
    sample_pair = pairs[0]
    pair_table = Table(title=f"Example pair {sample_pair.id}")
    pair_table.add_column("field", style="cyan")
    pair_table.add_column("text")
    pair_table.add_row("rejected_kind", sample_pair.rejected_kind.value)
    pair_table.add_row("chosen", sample_pair.chosen[:400])
    pair_table.add_row("rejected", sample_pair.rejected[:400])
    console.print(pair_table)

  Preference pairs by rejected  
              kind              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━┓
┃ rejected kind            ┃ n ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━┩
│ overreaching             │ 8 │
│ underreaching            │ 8 │
│ authority_misattribution │ 8 │
└──────────────────────────┴───┘

Total DPO rows: 24

                               Example pair dpo-ff9d1d00-overreaching                               
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ field         ┃ text                                                                             ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ rejected_kind │ overreaching                                                                     │
│ chosen        │ The SEC recommends safeguarding personal financial information such as Social    │
│               │ Security numbers, financial account numbers, phone numbers, e-mail addresses,    │
│               │ and usernames and passwords for online financial accounts.                       │
│ rejected      │ You should definitely invest in cybersecurity software to protect your online    │
│               │ investment accounts, as this will guarantee your information is safe.            │
└───────────────┴──────────────────────────────────────────────────────────────────────────────────┘

## 5. Optional judge filter and save

Set `VALIDATE_WITH_JUDGE=1` to keep a pair only when the judge prefers chosen over
rejected. Default is off to keep the demo cheap.

In [6]:
if VALIDATE_WITH_JUDGE:
    judge = create_judge_client()
    kept, dropped = filter_pairs_with_judge(judge, calibration_prompts, pairs)
    console.print(f"Judge kept {len(kept)} / {len(pairs)} pairs ({len(dropped)} dropped).")
    pairs = kept
else:
    console.print("Judge validation skipped (VALIDATE_WITH_JUDGE=0).")

save_typed_jsonl(
    DPO_CANDIDATES_PATH,
    calibration_prompts,
    to_dict=CalibrationPrompt.to_dict,
)
save_typed_jsonl(
    DPO_PAIRS_PATH,
    pairs,
    to_dict=PreferencePair.to_dict,
)
console.print(f"Wrote candidates → {DPO_CANDIDATES_PATH}")
console.print(f"Wrote pairs      → {DPO_PAIRS_PATH}")

Judge kept 22 / 24 pairs (2 dropped).

Wrote candidates → 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/synthetic/dpo_candidates
.jsonl

Wrote pairs      → 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/synthetic/dpo_preference
_pairs.jsonl

## 6. Optional LoRA DPO

Same CUDA / 4-bit LoRA constraints as notebook 05. Leave `RUN_DPO=0` on CPU or
Apple Silicon — you still have a usable preference JSONL.

The next cell records how the **un-adapted HF base** scores on `refusal_calibration`,
which is the fair comparison point for DPO. When `DPO_BASE_MODEL` matches
`SFT_BASE_MODEL` it reuses notebook 05's `data/results/hf_base_scores.json`;
otherwise it scores the base model here (needs CUDA).

**Direct Preference Optimization (DPO)** trains on **pairs**. For the same prompt \(x\), the model should assign higher probability to the **chosen** completion \(y_w\) than to the **rejected** one \(y_l\).

Classic RLHF first fits a **reward model**, then runs **PPO**. DPO skips both — the ranking is a classification-style loss on the policy, with a frozen **reference** model $\pi_{\text{ref}}$ (usually the model checkpoint) so the policy does not collapse or drift.

| | **SFT** | **RLHF** | **DPO** |
|--|--|--|--|
| Signal | one gold \(y\) | preferences → reward → PPO | preferences → one loss |
| Extra models | — | reward model + critic | frozen reference only |


![DPO chart](./images/DPO.png)


**Loss hyper-parameters** $ \sigma $ = sigmoid, $\beta$ = how hard to stay near $\pi_{\text{ref}}$: 

$$
\mathcal{L}_{\text{DPO}} = -\log\sigma\Big(\beta \log\frac{\pi_\theta(y_w\mid x)}{\pi_{\text{ref}}(y_w\mid x)} - \beta \log\frac{\pi_\theta(y_l\mid x)}{\pi_{\text{ref}}(y_l\mid x)}\Big)
$$

TRL’s `DPOTrainer` implements this.

**References**
- Rafailov et al., *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*, NeurIPS 2023. [arXiv:2305.18290](https://arxiv.org/abs/2305.18290)
- Hugging Face TRL — [DPO Trainer](https://huggingface.co/docs/trl/dpo_trainer)
- Ouyang et al., *Training language models to follow instructions with human feedback* (RLHF baseline). [arXiv:2203.02155](https://arxiv.org/abs/2203.02155)


In [7]:
# Load previously generated DPO pairs
pairs = load_typed_jsonl(DPO_PAIRS_PATH, PreferencePair.from_dict)
print(f"Number of DPO pairs: {len(pairs)}")

Number of DPO pairs: 22


In [8]:
from aieng.syn_data.text import (
    HF_BASE_SCORES_PATH,
    RESULTS_DIR,
    TEST_SET_PATH,
    FailureMode,
    QASample,
    create_judge_client,
    read_json,
    run_inference,
    save_baseline_results,
    score_predictions,
)
from aieng.syn_data.text.sft import Hf4BitInferenceClient

refusal_samples = [
    sample
    for sample in load_typed_jsonl(TEST_SET_PATH, QASample.from_dict)
    if sample.failure_mode == FailureMode.REFUSAL_CALIBRATION
]


def score_hf_base() -> dict[str, float]:
    """Score the un-adapted base model on the refusal slice (needs CUDA)."""
    client = Hf4BitInferenceClient(BASE_MODEL)
    predictions = run_inference(client, refusal_samples)
    scores = score_predictions(create_judge_client(), refusal_samples, predictions)
    summary = save_baseline_results(
        predictions,
        scores,
        refusal_samples,
        predictions_path=RESULTS_DIR / "dpo_base_predictions.jsonl",
        scores_path=RESULTS_DIR / "dpo_base_scores.json",
    )
    client.release()
    return summary["overall"]


# Notebook 05 already scored this model, but its "overall" spans every failure
# mode, so take the refusal_calibration slice.
if BASE_MODEL == SFT_BASE_MODEL and HF_BASE_SCORES_PATH.exists():
    source = f"reused from {HF_BASE_SCORES_PATH.name}"
    hf_base_refusal = read_json(HF_BASE_SCORES_PATH)["by_failure_mode"][
        "refusal_calibration"
    ]
else:
    source = f"scored here from {BASE_MODEL}"
    hf_base_refusal = score_hf_base()

table = Table(title=f"HF base before DPO — refusal_calibration ({source})")
table.add_column("Metric", style="cyan", no_wrap=True)
table.add_column("Score", justify="right", style="yellow")
for metric, score in hf_base_refusal.items():
    table.add_row(metric, f"{score:.3f}")
console.print(table)


      HF base before DPO —       
refusal_calibration (reused from 
      hf_base_scores.json)       
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Metric                ┃ Score ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ correctness           │ 3.553 │
│ coherence             │ 4.789 │
│ instruction_following │ 4.789 │
│ factual_plausibility  │ 4.000 │
│ average               │ 4.283 │
└───────────────────────┴───────┘

In [9]:
if RUN_DPO:
    adapter_path = train_lora_dpo(
        pairs,
        DPO_ADAPTER_DIR,
        base_model=BASE_MODEL,
        num_train_epochs=1.0,
    )
    console.print(f"[bold green]DPO LoRA adapter saved to {adapter_path}[/bold green]")
else:
    console.print(
        "[bold yellow]DPO training skipped[/bold yellow]\n"
        "Set [green]RUN_DPO=1[/green] on a [green]CUDA[/green] machine to fine-tune.\n"
        f"Preference pairs are ready at {DPO_PAIRS_PATH}"
    )

/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Tokenizing train dataset:   0%|          | 0/22 [00:00<?, ? examples/s][RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token 

Step,Training Loss


DPO LoRA adapter saved to 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/models/dpo_lora_adapter

## 7. Evaluate DPO on held-out refusal calibration

Load notebook 01's `test_set.jsonl`, keep only `failure_mode == refusal_calibration`, and score the **DPO LoRA adapter** with the same judge as notebooks 01 and 05.

Requires a saved adapter under `models/dpo_lora_adapter` (`RUN_DPO=1` in the previous cell). CUDA is needed for `PeftInferenceClient`.


In [10]:
from aieng.syn_data.text.sft import PeftInferenceClient
from aieng.syn_data.text import (
    BASELINE_SCORES_PATH,
    RESULTS_DIR,
    TEST_SET_PATH,
    FailureMode,
    QASample,
    create_judge_client,
    read_json,
    run_inference,
    save_baseline_results,
    score_predictions,
)

test_samples = load_typed_jsonl(TEST_SET_PATH, QASample.from_dict)
refusal_samples = [
    sample
    for sample in test_samples
    if sample.failure_mode == FailureMode.REFUSAL_CALIBRATION
]
console.print(
    f"Test set: {len(test_samples)} total, "
    f"{len(refusal_samples)} refusal_calibration"
)
if not refusal_samples:
    raise FileNotFoundError(
        f"No refusal_calibration rows in {TEST_SET_PATH}. Re-run notebook 01."
    )


Test set: 56 total, 19 refusal_calibration

In [11]:

adapter_ready = DPO_ADAPTER_DIR.exists() and any(DPO_ADAPTER_DIR.iterdir())
if adapter_ready:
    eval_client = PeftInferenceClient(DPO_ADAPTER_DIR, BASE_MODEL)
    eval_label = "DPO LoRA"
else:
    raise FileNotFoundError(
        f"RUN_DPO=1 but no adapter at {DPO_ADAPTER_DIR}. Re-run the train cell."
    )

dpo_predictions = run_inference(eval_client, refusal_samples)
judge = create_judge_client()
dpo_scores = score_predictions(judge, refusal_samples, dpo_predictions)

dpo_pred_path = RESULTS_DIR / "dpo_refusal_predictions.jsonl"
dpo_scores_path = RESULTS_DIR / "dpo_refusal_scores.json"
dpo_summary = save_baseline_results(
    dpo_predictions,
    dpo_scores,
    refusal_samples,
    predictions_path=dpo_pred_path,
    scores_path=dpo_scores_path,
)

table = Table(title=f"{eval_label} — refusal_calibration")
table.add_column("Metric", justify="left", style="cyan", no_wrap=True)
table.add_column("Score", justify="right", style="magenta")
for metric, score in dpo_summary["overall"].items():
    table.add_row(metric, f"{score:.3f}")
console.print(table)

ollama_refusal = (
    read_json(BASELINE_SCORES_PATH)["by_failure_mode"]["refusal_calibration"]
    if BASELINE_SCORES_PATH.exists()
    else {}
)


def fmt(val):
    return f"{val:.3f}" if isinstance(val, (int, float)) else "—"


def delta(new, old):
    return f"{new - old:+.3f}" if isinstance(old, (int, float)) else "—"


cmp = Table(title="Refusal calibration: Ollama vs HF base vs DPO")
cmp.add_column("Metric", style="cyan", no_wrap=True)
cmp.add_column("Ollama", justify="right", style="yellow")
cmp.add_column("HF base", justify="right", style="blue")
cmp.add_column(eval_label, justify="right", style="green")
cmp.add_column("Δ Ollama", justify="right", style="magenta")
cmp.add_column("Δ HF base", justify="right", style="magenta")
for metric, dpo_val in dpo_summary["overall"].items():
    ollama_val = ollama_refusal.get(metric)
    hf_val = hf_base_refusal.get(metric)
    cmp.add_row(
        metric,
        fmt(ollama_val),
        fmt(hf_val),
        fmt(dpo_val),
        delta(dpo_val, ollama_val),
        delta(dpo_val, hf_val),
    )
console.print(cmp)

console.print(f"Wrote {dpo_pred_path}")
console.print(f"Wrote {dpo_scores_path}")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 713.16it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2026-08-27 22:50:48,609 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-sec_investor_bulletin::p0004-0 (answer length: 927)
2026-08-27 22:50:50,081 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-sec_investor_bulletin::p0004-2 (answer length: 599)
2026-08-27 22:50:51,083 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-sec_investor_bulletin::p0005-0 (answer length: 112)
2026-08-27 22:50:52,345 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-sec_investor_bulletin::p0005-2 (answer length: 649)
2026-08-27 22:50:53,513 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-sec_investor_bulletin::p0006-0 (answer length: 120)
2026-08-27 22:50:54,451 INFO aieng.syn_data.text.judg

 DPO LoRA — refusal_calibration  
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Metric                ┃ Score ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ correctness           │ 3.421 │
│ coherence             │ 4.789 │
│ instruction_following │ 4.737 │
│ factual_plausibility  │ 3.895 │
│ average               │ 4.211 │
└───────────────────────┴───────┘

                Refusal calibration: Ollama vs HF base vs DPO                 
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Metric                ┃ Ollama ┃ HF base ┃ DPO LoRA ┃ Δ Ollama ┃ Δ HF base ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━┩
│ correctness           │  4.053 │   3.553 │    3.421 │   -0.632 │    -0.132 │
│ coherence             │  4.895 │   4.789 │    4.789 │   -0.105 │    +0.000 │
│ instruction_following │  4.684 │   4.789 │    4.737 │   +0.053 │    -0.053 │
│ factual_plausibility  │  4.500 │   4.000 │    3.895 │   -0.605 │    -0.105 │
│ average               │  4.533 │   4.283 │    4.211 │   -0.322 │    -0.072 │
└───────────────────────┴────────┴─────────┴──────────┴──────────┴───────────┘

Wrote 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/results/dpo_refusal_pred
ictions.jsonl

Wrote 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/results/dpo_refusal_scor
es.json